# 09 · Consignment features: learned on LEMIS shipments, computed on CITES permits

**The question.** This is S2.2 in [`SYSTEM CODE/DESIGN_QUESTIONS.md`](../DESIGN_QUESTIONS.md). WildGuard describes each record on its own and against its peers, but not against the other items travelling with it. Both datasets group records into consignments:
- a LEMIS shipment (`control_number`) is one declaration: its line items share an exporter, date, port and companies (report §3.2; notebook 01, L1);
- a CITES export permit groups the records traded under it, and is present on 97% of records (notebook 05, M2).

So features that describe a record's consignment can be learned from LEMIS outcomes and computed on CITES. If they help, the limitation "items travelling on one permit cannot be learned" (ALIGNING §2.3; report §8 item 7) narrows.

**Candidate features**, computed per consignment and attached to each of its records:
- `cons_lines`: number of line items (records) in the consignment;
- `cons_genera`: number of distinct genera;
- `cons_groups`: number of distinct product groups;
- `cons_purposes`, `cons_sources`: number of distinct purposes and sources.

**A difference to keep in mind.** A LEMIS shipment also holds items of species CITES does not list, while a CITES permit holds only listed specimens, and each CITES record is one country's report. The first test checks whether that difference makes the features mean something else in each dataset.

**Fixed before looking:**
1. **Consignments.** LEMIS: all line items of a shipment. CITES: exporter-reported records of 2015–2024 sharing an export permit; a record without a permit is its own consignment.
2. **(a) Comparable.** For each feature, the population stability index (PSI) between LEMIS line items of listed species (training years) and CITES exporter-reported records, over the bins 1, 2, 3–5, 6–10, 11–50 and more than 50. A feature with PSI of 0.25 or more (the conventional threshold for a large shift) is dropped before (b).
3. **(b) Useful.** XGBoost (settings of `wildguard/models.py`, three seeds, scores averaged) trained on animals with a class 2000–2009, early stopping on 2010 listed line items, evaluated on 2011 (the selection year, DESIGN_QUESTIONS B1): the shared features alone against the shared features plus the comparable consignment features.
4. **Rule:** the comparable features are adopted only if PR-AUC on listed 2011 line items improves with a 95% paired-bootstrap interval (resampling shipments) above zero.

In [1]:
import time

import duckdb
import numpy as np
import pandas as pd

import eda_setup as E
import experiment_setup as X_
from wildguard import config
from wildguard.features import SHARED_FEATURES, FeatureBuilder, to_model_matrix

E.style()
t0 = time.time()
CONS = ["cons_lines", "cons_genera", "cons_groups", "cons_purposes", "cons_sources"]
AGG = """count(*) AS cons_lines, count(DISTINCT genus) AS cons_genera,
         count(DISTINCT product_group) AS cons_groups, count(DISTINCT purpose) AS cons_purposes,
         count(DISTINCT source) AS cons_sources"""
con = duckdb.connect()
con.execute("PRAGMA threads=8")
lemis_cons = con.execute(f"""SELECT shipment_id, {AGG} FROM '{config.LEMIS_PARQUET}'
                             GROUP BY 1""").df()
cites_cons = con.execute(f"""
    WITH r AS (SELECT *, coalesce(export_permit, 'record ' || shipment_id) AS consignment
               FROM '{config.CITES_PARQUET}'
               WHERE year BETWEEN 2015 AND 2024 AND reporter_type = 'E')
    SELECT {", ".join(f"c.{f}" for f in CONS)}
    FROM r JOIN (SELECT consignment, {AGG} FROM r GROUP BY 1) c USING (consignment)""").df()
df = X_.load_lemis()
df = df[df["animal"]].merge(lemis_cons, on="shipment_id", how="left").reset_index(drop=True)
print(f"{len(df):,} LEMIS line items, {len(cites_cons):,} CITES exporter-reported records "
      f"({time.time() - t0:.0f}s)")

5,107,119 LEMIS line items, 4,835,543 CITES exporter-reported records (8s)


## 1. (a) Do the features mean the same thing in both datasets?

In [2]:
BINS = [0, 1, 2, 5, 10, 50, np.inf]
LABELS = ["1", "2", "3–5", "6–10", "11–50", ">50"]


def dist(x):
    return pd.cut(x, BINS, labels=LABELS).value_counts(normalize=True).reindex(LABELS).fillna(0)


lemis_ref = df[df.sp_listed & (df.year <= 2010)]
rows, tables = [], {}
for f in CONS:
    p, q = dist(lemis_ref[f]), dist(cites_cons[f])
    psi = float(((p - q) * np.log((p + 1e-6) / (q + 1e-6))).sum())
    rows.append({"feature": f, "PSI": psi, "comparable": psi < 0.25,
                 "LEMIS median": lemis_ref[f].median(), "CITES median": cites_cons[f].median()})
    tables[f] = pd.DataFrame({"LEMIS listed": p, "CITES": q})
comp = pd.DataFrame(rows).set_index("feature")
COMPARABLE = comp.index[comp.comparable].tolist()
display(comp.style.format({"PSI": "{:.3f}"}))
pd.concat(tables, axis=1).style.format("{:.1%}")

,PSI,comparable,LEMIS median,CITES median
feature,,,,
cons_lines,0.576,False,7.000000,2.000000
cons_genera,0.719,False,3.000000,1.000000
cons_groups,0.290,False,1.000000,1.000000
cons_purposes,0.113,True,1.000000,1.000000
cons_sources,0.165,True,1.000000,1.000000


## 2. (b) Do the comparable features improve the model?

In [3]:
yr, y, listed = df["year"].to_numpy(), df["label"].to_numpy(), df["sp_listed"].to_numpy()
tr, va, ev = yr <= 2009, (yr == 2010) & listed, yr == 2011
fb = FeatureBuilder(use_value=False).fit(df[tr])
F = {k: fb.transform(df[m]).reset_index(drop=True) for k, m in (("tr", tr), ("va", va), ("ev", ev))}
for k, m in (("tr", tr), ("va", va), ("ev", ev)):
    for f in CONS:
        F[k][f] = df.loc[m, f].to_numpy()
sets = {"shared features": SHARED_FEATURES, "+ consignment features": SHARED_FEATURES + COMPARABLE}
s = {}
for name, feats in sets.items():
    X_tr, cats = to_model_matrix(F["tr"], feats)
    X_va = to_model_matrix(F["va"], feats, cats)[0]
    X_ev = to_model_matrix(F["ev"], feats, cats)[0]
    s[name] = np.mean([X_.fit_split(X_tr, y[tr], X_va, y[va], seed).predict_proba(X_ev)[:, 1]
                       for seed in (0, 1, 2)], axis=0)
    print(f"{name}: trained ({time.time() - t0:.0f}s)")
ev_df = df[ev].reset_index(drop=True)
out = []
for subset, mask in (("listed species", ev_df.sp_listed.to_numpy()), ("all", np.ones(len(ev_df), bool))):
    out.append(X_.pr_auc_bootstrap(ev_df.label.to_numpy()[mask], {k: v[mask] for k, v in s.items()},
                                   ev_df.shipment_id.to_numpy()[mask],
                                   [("+ consignment features", "shared features")])
               .assign(**{"line items": subset}))
res = pd.concat(out).set_index("line items")[["pr_auc_b", "pr_auc_a", "difference", "low_95", "high_95"]]
res.columns = ["PR-AUC, shared", "PR-AUC, + consignment", "difference", "95% low", "95% high"]
res.style.format("{:.3f}")

shared features: trained (117s)


+ consignment features: trained (205s)


,"PR-AUC, shared","PR-AUC, + consignment",difference,95% low,95% high
line items,,,,,
listed species,0.548,0.550,0.002,0.000,0.005
all,0.623,0.624,0.001,-0.000,0.002


## 3. The rule

In [4]:
ADOPT = bool(COMPARABLE) and res.loc["listed species", "95% low"] > 0
print("Comparable features:", COMPARABLE or "none")
print("Decision:", f"adopt {COMPARABLE}" if ADOPT else "consignment features are not adopted")

Comparable features: ['cons_purposes', 'cons_sources']
Decision: adopt ['cons_purposes', 'cons_sources']


## Findings

| # | Finding | Consequence |
|---|---|---|
| K1 | The number of line items, genera and product groups per consignment mean different things in the two datasets (PSI 0.29–0.72). A LEMIS shipment carrying a listed species has a median of 7 line items and 3 genera, because it also holds unlisted items; a CITES permit has a median of 2 records and 1 genus | Not used |
| K2 | Every LEMIS shipment of the training years has a single purpose (1,493,154 shipments), so the number of purposes is constant in training and carries nothing to learn, although it passes the comparability test | Dropped: the model never splits on a constant |
| K3 | The number of sources is comparable (PSI 0.165): 25% of listed LEMIS line items travel in a shipment with mixed sources, 10% of CITES records on a permit with mixed sources | Candidate |
| K4 | Adding the comparable features raises PR-AUC on listed 2011 line items by 0.002, with a 95% interval whose lower end lies just above zero (it rounds to 0.000); on all line items the gain (0.001) is not significant. The gain comes from the number of sources | Passes the rule, narrowly |

**Decision (S2.2):** adopt `cons_sources`, the number of distinct sources in a record's consignment (a LEMIS shipment; a CITES export permit). The gain is small, and feature selection (DESIGN_QUESTIONS B8) tests the feature again. The limitation that "items travelling on one permit cannot be learned" narrows only slightly: a mix of sources within a consignment can be learned; the two countries' reports and re-export chains still cannot.